# Load Battery Dataset

In [ ]:
import os
os.chdir('..')
import json
import pandas as pd
import glob

files = glob.glob('data/raw/FastCharge*.json')
print(f"Found {len(files)} files")

In [ ]:
import sys
sys.path.insert(0, 'scripts')
from dedupe_policy import is_kept_file

cells = []

for i, file in enumerate(files):
    try:
        file_id = os.path.basename(file)
        if not is_kept_file(file_id):
            continue

        print(f"Processing {i}: {file}")
        
        with open(file) as f:
            data = json.load(f)
        
        summary = pd.DataFrame(data['summary'])
        summary = summary[summary['cycle_index'] >= 1]
        
        if len(summary) == 0:
            continue
        
        initial_cap = summary['discharge_capacity'].iloc[0]
        threshold = 0.8 * initial_cap
        below_threshold = summary[summary['discharge_capacity'] < threshold]
        
        if len(below_threshold) > 0:
            eol = below_threshold.iloc[0]['cycle_index']
        else:
            eol = summary['cycle_index'].iloc[-1]
        
        cells.append({
            'file_id': file_id,
            'cell_id': data['barcode'],
            'EOL': eol,
            'initial_capacity': initial_cap
        })
    except Exception as e:
        print(f"ERROR on {file}: {e}")
        break

print(f"Processed {len(cells)} cells (deduped, Week 2 policy)")

In [ ]:
dataset = pd.DataFrame(cells)
dataset.to_csv('data/cell_targets.csv', index=False)
print("Saved to data/cell_targets.csv")
print(dataset.head())